In [8]:
import os
import cv2 as cv
import numpy as np
import pandas as pd

In [ ]:
image_folder = "dataset/100"
output_folder = "rois/100"  # Folder to save ROI images
os.makedirs(output_folder, exist_ok=True)


In [10]:
# Function to extract ROI
def extract_roi(image, coords):
    x1, y1 = coords[0]
    x2, y2 = coords[1]
    return image[y1:y2, x1:x2]

In [11]:
# Function to compute histogram
def compute_histogram(image):
    hist = cv.calcHist([image], [0], None, [256], [0, 256])
    return hist.flatten()  # Convert to 1D array


In [12]:
# Define ROI coordinates
security_mark_coords = [(0, 56), (32, 150)]
green_strip_coords = [(380, 0), (410, 300)]
serial_number_coords = [(445, 240), (630, 300)]
gandhiji_coords = [(152, 65), (385, 300)]


In [13]:
data = []

# Process each image
image_files = [f for f in os.listdir(image_folder) if f.endswith(('.jpg', '.png', '.jpeg'))]

for img_name in image_files:
    image_path = os.path.join(image_folder, img_name)
    img = cv.imread(image_path)

    if img is None:
        print(f"Error loading {img_name}")
        continue
    # Preprocessing steps
    rescaled_img = cv.resize(img, (700, 300))  # Resize to fixed size
    gray_img = cv.cvtColor(rescaled_img, cv.COLOR_BGR2GRAY)
    blurred_img = cv.GaussianBlur(gray_img, (5, 5), 0)
    equalized_img = cv.equalizeHist(blurred_img)
    edge_img = cv.Canny(equalized_img, 150, 255)

    # Extract ROIs
    security_mark = extract_roi(edge_img, security_mark_coords)
    green_strip = extract_roi(edge_img, green_strip_coords)
    serial_number = extract_roi(edge_img, serial_number_coords)
    gandhiji = extract_roi(edge_img, gandhiji_coords)

    # Compute histograms
    security_mark_hist = compute_histogram(security_mark)
    green_strip_hist = compute_histogram(green_strip)
    serial_number_hist = compute_histogram(serial_number)
    gandhiji_hist = compute_histogram(gandhiji)

    # Save ROIs as images
    sec_mark_path = os.path.join(output_folder, f"{img_name}_security_mark.jpg")
    green_strip_path = os.path.join(output_folder, f"{img_name}_green_strip.jpg")
    serial_number_path = os.path.join(output_folder, f"{img_name}_serial_number.jpg")
    gandhiji_path = os.path.join(output_folder, f"{img_name}_gandhiji.jpg")

    cv.imwrite(sec_mark_path, security_mark)
    cv.imwrite(green_strip_path, green_strip)
    cv.imwrite(serial_number_path, serial_number)
    cv.imwrite(gandhiji_path, gandhiji)

        # Append data to list
    data.append({
        "ID": img_name,
        "Security_Mark_Hist": security_mark_hist.tolist(),
        "Green_Strip_Hist": green_strip_hist.tolist(),
        "Serial_Number_Hist": serial_number_hist.tolist(),
        "Gandhiji_Hist": gandhiji_hist.tolist(),
        "Security_Mark_Image": sec_mark_path,
        "Green_Strip_Image": green_strip_path,
        "Serial_Number_Image": serial_number_path,
        "Gandhiji_Image": gandhiji_path
    })

In [14]:
# Convert to DataFrame
df = pd.DataFrame(data)

# Save to Excel
excel_path = "extracted_features_200.xlsx"
df.to_excel(excel_path, index=False)

print(f"Data successfully saved in {excel_path} for {len(image_files)} images!")

Data successfully saved in extracted_features_200.xlsx for 20 images!
